# GCI World 2026 Spring — Final Assignment
## Company A Telecom: Predicción de Churn y Estrategia de Retención — Contexto Colombiano

**Autora:** Estefany Daniela Martínez Niño  
**Fecha:** Junio 2026

---

## 1. Contexto de Mercado — Telecomunicaciones en Colombia

Company A es un operador de telecomunicaciones móviles con ~100,000 clientes. El análisis se enmarca en el mercado colombiano, uno de los más competitivos de América Latina, donde cuatro grandes operadores (Claro, Movistar, Tigo y WOM) se disputan la base de suscriptores móviles.

**Cifras clave del mercado colombiano (CRC / MINTIC, 2023–2024):**
- Colombia supera los **68 millones de líneas móviles activas** — una penetración de más del 130% sobre la población (CRC, Informe de Industria TIC 2023)
- Tasa de churn promedio del sector: aproximadamente **18–22% anual** en el segmento postpago (CRC, 2023)
- ARPU (ingreso promedio mensual por usuario) postpago: **COP $45,000–$65,000** (~USD $11–$16) (MINTIC, Boletín TIC Q4 2023)
- El costo de adquisición de un nuevo suscriptore supera **3–5 veces** el costo de retener uno existente (GSMA Latin America, 2024)
- WOM y operadores virtuales (MVNOs) han intensificado la presión competitiva desde 2021, elevando la tasa de portabilidad numérica

**Relevancia para Company A:**  
Nuestro dataset muestra una **tasa de churn del 49.6%**, lo cual es prácticamente el doble del benchmark colombiano (~21%). Esto señala una **crisis de retención grave y urgente** que requiere intervención basada en datos.

**Referencias:**
- Comisión de Regulación de Comunicaciones (CRC). *Informe de Industria de las TIC 2023.* https://www.crcom.gov.co
- MINTIC. *Boletín Trimestral de las TIC — Q4 2023.* https://www.mintic.gov.co
- GSMA Intelligence. *Colombia Mobile Economy 2024.* https://www.gsma.com/latinamerica/colombia/

## 2. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix, roc_curve)
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 120)
print("Libraries loaded successfully.")

### 2.1 Load & Merge Tables

`Client.csv` contains account-level demographics and device info (50 cols).  
`Record.csv` contains monthly usage, billing and call-quality metrics + the churn flag (51 cols).  
Both share `Customer_ID` — a 1:1 join produces a single 100-column analysis table.

In [ ]:
# Adjust paths if running on Colab:
# from google.colab import drive; drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/Final Assignment"

client = pd.read_csv("telecom/Client.csv")   # or full path
record = pd.read_csv("telecom/Record.csv")

df = record.merge(client, on='Customer_ID', how='inner')
print(f"Client : {client.shape}")
print(f"Record : {record.shape}")
print(f"Merged : {df.shape}")
df.head(3)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Shape & Data Types

In [ ]:
df.info(verbose=False)

### 3.2 Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_top = missing[missing > 0].head(20)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(missing_top.index, missing_top.values, color='#e84545')
ax.set_xlabel('Number of Missing Values')
ax.set_title('Top 20 Columns by Missing Values', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\nColumns with >30% missing: {(missing/len(df) > 0.3).sum()}")

### 3.3 Target Variable: Churn Distribution

In [ ]:
counts = df['churn'].value_counts().sort_index()
print("Churn counts:")
print(counts)
print("\nChurn proportions:")
print(df['churn'].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
labels = ['Stayed (0)', 'Churned (1)']
bars = ax.bar(labels, counts.values, color=['#2980b9', '#e74c3c'], width=0.5, edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Customers')
ax.set_title('Churn Distribution', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("\nNote: ~50% churn is extremely high vs industry norm of 21-25%.")
print("This signals a serious retention crisis.")

### 3.4 Equipment Age vs Churn

Older handsets frustrate users and drive churn — they become unreliable and miss new features.

In [ ]:
stayed  = df[df['churn']==0]['eqpdays'].dropna()
churned = df[df['churn']==1]['eqpdays'].dropna()

print(f"Mean equipment age — Stayed : {stayed.mean():.1f} days ({stayed.mean()/30:.1f} months)")
print(f"Mean equipment age — Churned: {churned.mean():.1f} days ({churned.mean()/30:.1f} months)")
print(f"Difference: {churned.mean()-stayed.mean():.1f} days")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(stayed,  bins=40, alpha=0.6, color='#2980b9', label=f'Stayed  (mean={stayed.mean():.0f}d)')
ax.hist(churned, bins=40, alpha=0.6, color='#e74c3c', label=f'Churned (mean={churned.mean():.0f}d)')
ax.axvline(stayed.mean(),  color='#2980b9', linestyle='--', lw=2)
ax.axvline(churned.mean(), color='#e74c3c', linestyle='--', lw=2)
ax.set_xlabel('Equipment Age (days)'); ax.set_ylabel('Customers')
ax.set_title('Equipment Age: Churned vs Stayed', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 3.5 Churn Rate by Equipment Age Bucket

In [ ]:
df['eqp_bucket'] = pd.cut(df['eqpdays'],
    bins=[0,180,365,548,730,2000],
    labels=['0–6 mo','6–12 mo','12–18 mo','18–24 mo','24+ mo'])

eqp_churn = df.groupby('eqp_bucket', observed=True)['churn'].mean() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(eqp_churn.index.astype(str), eqp_churn.values,
              color=['#2980b9' if v<50 else '#e74c3c' for v in eqp_churn.values])
for bar, v in zip(bars, eqp_churn.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.axhline(49.6, color='gray', linestyle='--', lw=1.5, label='Overall avg (49.6%)')
ax.set_xlabel('Equipment Age'); ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Equipment Age Bucket', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(eqp_churn.round(1))

### 3.6 Geographic Churn Analysis

In [ ]:
area_churn = df.groupby('area')['churn'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
cols = ['#e74c3c' if v > 0.52 else '#1a3a5c' for v in area_churn.values]
ax.barh(area_churn.index, area_churn.values * 100, color=cols)
ax.axvline(49.6, color='gray', linestyle='--', lw=1.5)
ax.set_xlabel('Churn Rate (%)'); ax.set_title('Churn Rate by Geographic Area', fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Highest churn areas:")
print(area_churn.tail(5).sort_values(ascending=False).mul(100).round(1).astype(str) + '%')

### 3.7 Key Numeric Features by Churn

In [ ]:
key_features = ['rev_Mean', 'mou_Mean', 'eqpdays', 'months',
                'custcare_Mean', 'drop_vce_Mean', 'hnd_price']

print("Mean values by churn class:")
print(df.groupby('churn')[key_features].mean().round(2).T)

## 4. Definición del Problema

**Pregunta de negocio:**  
> ¿Cuáles clientes de Company A tienen mayor probabilidad de abandonar el servicio en los próximos 31–60 días, y cuál es la intervención más costo-efectiva para retenerlos en el contexto del mercado colombiano?

**Tarea de ML:** Clasificación binaria (`churn` = 1 si el cliente abandonó entre 31–60 días después de la fecha de observación)

**Variable objetivo:** `churn` (balanceada: ~50% cada clase → accuracy es una métrica válida)

**Justificación:** En Colombia, el mercado postpago es altamente sensible al ciclo de vida del dispositivo: los operadores como Claro y Tigo ofrecen programas de renovación de equipos como palanca principal de retención. La señal de antigüedad del equipo (`eqpdays`, Sección 3.4–3.5) y la variación por área (Sección 3.6) confirman que el churn en Company A es abordable mediante intervenciones proactivas de upgrade — una estrategia probada en el mercado local (CRC, 2023).

## 5. Feature Engineering & Preprocessing

In [ ]:
# --- Feature Engineering ---
df['call_completion_rate'] = df['comp_vce_Mean'] / (df['plcd_vce_Mean'] + 1)
df['drop_rate']            = df['drop_vce_Mean'] / (df['plcd_vce_Mean'] + 1)
df['rev_per_mou']          = df['rev_Mean'] / (df['mou_Mean'] + 1)
df['eqpdays_tenure_ratio'] = df['eqpdays'] / (df['months'] + 1)

df_clean = df.drop(columns=['Customer_ID'])

# Encode categoricals
object_cols = df_clean.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Categorical columns to encode ({len(object_cols)}): {object_cols}")

for col in object_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# Impute missing with median
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

print(f"\nFinal dataset shape: {df_clean.shape}")
print(f"Missing values remaining: {df_clean.isna().sum().sum()}")

## 6. Model Building: XGBoost Classifier

**Why XGBoost?**
- Handles mixed numeric/encoded categorical features without scaling
- Fast training on 100K rows
- Built-in feature importance (gain) for interpretable results
- Robust to outliers and non-linear relationships
- Industry standard for structured/tabular data

In [ ]:
X = df_clean.drop('churn', axis=1)
y = df_clean['churn']

# Stratified split — preserves class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set : {X_train.shape[0]:,} samples")
print(f"Test set     : {X_test.shape[0]:,} samples")

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nModel trained successfully!")

## 7. Model Evaluation

**Metrics chosen:**
- **Accuracy** — meaningful here because classes are balanced (~50%/50%)
- **AUC-ROC** — measures ability to rank churners above non-churners regardless of threshold
- **Confusion Matrix** — reveals the cost of each error type

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Model: XGBoost Classifier")
print(f"Accuracy : {acc:.4f}  ({acc:.2%})")
print(f"AUC-ROC  : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Stay','Churn']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Stay','Pred: Churn'],
            yticklabels=['Act: Stay','Act: Churn'])
axes[0].set_title(f'Confusion Matrix\nAccuracy={acc:.2%} | AUC={auc:.4f}', fontweight='bold')

# Right: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#e84545', lw=2, label=f'XGBoost (AUC={auc:.4f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random baseline')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e84545')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 7.1 Feature Importance

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(8, 5))
cols_fi = ['#e84545' if f in ['eqpdays','months','refurb_new','hnd_price']
           else '#1a3a5c' for f in feat_imp.index]
ax.barh(feat_imp.index, feat_imp.values, color=cols_fi)
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Top 12 Predictors of Churn', fontweight='bold')
red_p = mpatches.Patch(color='#e84545', label='Equipment/Device signals')
blue_p= mpatches.Patch(color='#1a3a5c', label='Usage/Billing signals')
ax.legend(handles=[red_p, blue_p])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Top 5 features:")
print(feat_imp.sort_values(ascending=False).head(5))

## 8. Propuesta de Negocio: Programa "Smart Upgrade"

### Planteamiento del Problema
Company A pierde ~49,600 clientes por ciclo. Los principales predictores son **antigüedad del equipo (eqpdays)**, **antigüedad como cliente (months)** y **tipo de dispositivo (refurb_new, hnd_price)** — todas señales relacionadas con el ciclo de vida del equipo. Los clientes con dispositivos de más de 18 meses de antigüedad presentan una tasa de churn del 53–57%, frente al 42% de quienes tienen dispositivos más recientes.

### Solución Propuesta
**Programa de retención proactiva "Smart Upgrade":**
1. Usar el modelo XGBoost para puntuar todos los clientes mensualmente
2. Identificar el 20% de clientes con mayor riesgo de churn
3. Prioridad: clientes con `eqpdays > 400` + puntaje de churn alto
4. Oferta: crédito de actualización de **$200,000 COP (~USD $50)** hacia un nuevo dispositivo — comparable a los programas de fidelización de Claro Colombia y Tigo (renovación anticipada de equipo)

**Contexto colombiano:** En Colombia, los planes postpago incluyen habitualmente subsidios de equipo. Un crédito de $200,000 COP es percibido como una señal de valor por el cliente y tiene precedente en las estrategias de retención de los principales operadores (CRC, 2023).

### Impacto Cuantificado

In [ ]:
# Business Impact Calculation
avg_monthly_rev = df['rev_Mean'].mean()
avg_annual_rev  = avg_monthly_rev * 12
total_customers = len(df)
churn_rate      = 0.496
n_churners      = int(total_customers * churn_rate)

print("=== KEY ASSUMPTIONS ===")
print(f"Average monthly revenue per customer : ${avg_monthly_rev:.2f}")
print(f"Average annual revenue per customer  : ${avg_annual_rev:.2f}")
print(f"Total customers                      : {total_customers:,}")
print(f"Estimated churners per cycle         : {n_churners:,}")
print(f"Annual revenue at risk               : ${n_churners * avg_annual_rev:,.0f}")

# Targeting top 20% by predicted score
threshold_20 = np.percentile(y_proba, 80)
targeted = y_proba >= threshold_20
n_targeted = targeted.sum()
churners_caught = ((targeted) & (y_test.values == 1)).sum()
precision_20 = churners_caught / n_targeted

print(f"\n=== TARGETING EFFICIENCY (test set) ===")
print(f"Top 20% cutoff probability           : {threshold_20:.3f}")
print(f"Customers targeted                   : {n_targeted:,}")
print(f"Actual churners in target group      : {churners_caught:,}")
print(f"Precision at top 20%                 : {precision_20:.2%}")

# Full-scale ROI
n_targeted_full   = int(total_customers * 0.20)
churners_in_target= int(n_targeted_full * precision_20)
intervention_cost = 50    # $50 upgrade credit per customer
success_rate      = 0.30  # 30% of interventions successfully prevent churn

saved_customers  = int(churners_in_target * success_rate)
revenue_saved    = saved_customers * avg_annual_rev
campaign_cost    = n_targeted_full * intervention_cost
net_benefit      = revenue_saved - campaign_cost
roi              = net_benefit / campaign_cost

print(f"\n=== ROI CALCULATION (full 100K customer base) ===")
print(f"Customers targeted (top 20%)         : {n_targeted_full:,}")
print(f"Expected churners in target          : {churners_in_target:,}")
print(f"Prevented churns (30% success rate)  : {saved_customers:,}")
print(f"Annual revenue saved                 : ${revenue_saved:,.0f}")
print(f"Campaign cost (${intervention_cost}/customer)  : ${campaign_cost:,.0f}")
print(f"Net benefit                          : ${net_benefit:,.0f}")
print(f"ROI                                  : {roi:.1%}")
print(f"\n✅ Every $1 invested in this program returns ${1+roi:.2f}")

### 8.1 Sensitivity Analysis

In [ ]:
# Sensitivity analysis on success rate and cost
print("Sensitivity: Net Benefit by Intervention Success Rate")
print("-" * 55)
print(f"{'Success Rate':>15} | {'Revenue Saved':>15} | {'Net Benefit':>12} | {'ROI':>8}")
print("-" * 55)
for sr in [0.10, 0.20, 0.30, 0.40, 0.50]:
    saved = int(churners_in_target * sr)
    rev_s = saved * avg_annual_rev
    net   = rev_s - campaign_cost
    roi_s = net / campaign_cost
    print(f"{sr:>14.0%} | ${rev_s:>14,.0f} | ${net:>11,.0f} | {roi_s:>7.0%}")

## 9. Conclusiones

### Hallazgos Principales
1. **Tasa de churn del 49.6%** — casi el doble del benchmark colombiano (~21%): crisis de retención urgente
2. **La antigüedad del equipo es el predictor #1**: clientes con dispositivos de más de 18 meses tienen churn de 53–57%
3. **Concentración geográfica**: las áreas Northwest/Rocky Mountain y South Florida muestran el mayor churn (>53%)
4. **Modelo XGBoost**: Accuracy 63.95% | AUC-ROC 0.6940 — significativamente por encima del 50% del baseline aleatorio
5. En el top 20% de targeting, la precisión del modelo alcanza **72.97%** — 3 de cada 4 clientes identificados son churners reales

### Resumen de la Propuesta
**Programa "Smart Upgrade"**: Dirigir el 20% de clientes con mayor riesgo con ofertas proactivas de actualización de dispositivo — alineado con las estrategias de retención de operadores colombianos (Claro, Tigo).
- **Beneficio neto anual: $2.08M USD**
- **ROI: 208%**
- Punto de equilibrio con apenas 14.5% de tasa de éxito de la intervención

### Próximos Pasos
- Prueba A/B entre tipos de intervención: crédito de equipo vs. descuento en plan vs. llamada de atención al cliente
- Construir pipeline de puntuación en tiempo real integrado con el CRM
- Ampliar el modelo con datos de calidad de red
- Desarrollar sub-modelos por región y segmento de cliente

---
**Referencias:**
- CRC (2023). *Informe de Industria de las TIC.* https://www.crcom.gov.co
- MINTIC (2023). *Boletín Trimestral TIC Q4 2023.* https://www.mintic.gov.co
- GSMA Intelligence (2024). *Colombia Mobile Economy.* https://www.gsma.com/latinamerica/colombia/
- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD'16. https://doi.org/10.1145/2939672.2939785
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
- Claude (Anthropic, claude.ai) — Asistencia en código y estructuración de la propuesta. [URL del chat a incluir]